In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import random
import copy
import time

from train.data_loader import load_data
from train.knn_algorithm import preprocess_data, knn_algorithm
from train.proximity_algorithm import preprocess_data_proximity, proximity_algorithm
from train.evaluation import evaluate_results
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

RANDOM_SEED = 23
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

def simulate_beacon_anomalies(data, num_anomalies, beacon_columns):

    modified_data = copy.deepcopy(data)

    if num_anomalies > 0:
        if num_anomalies > len(beacon_columns):
            num_anomalies = len(beacon_columns)

        malfunctioning_beacons = random.sample(beacon_columns, num_anomalies)

        for beacon in malfunctioning_beacons:
            modified_data[beacon] = 110.0

    return modified_data

def run_experiment_with_anomalies(
    train_data, test_data, num_anomalies, beacon_columns,
    rf_champion, lr_champion_pipe,
    X_train, y_train_artwork_encoded, y_test_artwork_encoded
):
    # Simula le anomalie dei beacon sui dati di test
    modified_test_data_df = simulate_beacon_anomalies(test_data, num_anomalies, beacon_columns)

    # Definisci il valore di default per non rilevati
    default_non_detected_value = 110.0

    # Dizionario per memorizzare i risultati
    results = {
        'proximity': {},
        'knn_k1': {},
        'knn_k57': {},
        'rf_champion': {},
        'lr_champion': {}
    }

    # Esegui Algoritmo Proximity (Baseline)
    preprocessed_data = preprocess_data_proximity(
        train_data,
        modified_test_data_df,
        default_non_detected_value,
        None
    )
    proximity_results, proximity_detailed_results = proximity_algorithm(
        preprocessed_data,
        default_non_detected_value
    )
    proximity_results = evaluate_results(proximity_results, proximity_detailed_results)
    results['proximity'] = {
        'room_accuracy': proximity_results['accuracy']['room'],
        'artwork_top1_accuracy': proximity_results['accuracy']['artwork_top1'],
        'artwork_top3_accuracy': proximity_results['accuracy']['artwork_top3']
    }

    # Esegui Algoritmo KNN k=1 (Baseline)
    preprocessed_data = preprocess_data(
        train_data,
        modified_test_data_df,
        default_non_detected_value,
        None
    )
    knn_k1_results, knn_k1_detailed_results = knn_algorithm(
        preprocessed_data, 1, 'cityblock', default_non_detected_value
    )
    knn_k1_results = evaluate_results(knn_k1_results, knn_k1_detailed_results)
    results['knn_k1'] = {
        'room_accuracy': knn_k1_results['accuracy']['room'],
        'artwork_top1_accuracy': knn_k1_results['accuracy']['artwork_top1'],
        'artwork_top3_accuracy': knn_k1_results['accuracy']['artwork_top3']
    }

    # Esegui Algoritmo KNN k=57 (Baseline)
    knn_k57_results, knn_k57_detailed_results = knn_algorithm(
        preprocessed_data, 57, 'cityblock', default_non_detected_value
    )
    knn_k57_results = evaluate_results(knn_k57_results, knn_k57_detailed_results)
    results['knn_k57'] = {
        'room_accuracy': knn_k57_results['accuracy']['room'],
        'artwork_top1_accuracy': knn_k57_results['accuracy']['artwork_top1'],
        'artwork_top3_accuracy': knn_k57_results['accuracy']['artwork_top3']
    }

    X_test_anomalous = modified_test_data_df[beacon_columns]
    y_pred_rf = rf_champion.predict(X_test_anomalous)
    rf_acc_top1 = accuracy_score(y_test_artwork_encoded, y_pred_rf)

    # Calcola Top-3 per RF
    rf_probs = rf_champion.predict_proba(X_test_anomalous)
    rf_top3 = rf_probs.argsort(axis=1)[:, -3:]
    rf_correct_top3 = [y_test_artwork_encoded[i] in rf_top3[i] for i in range(len(y_test_artwork_encoded))]
    rf_acc_top3 = sum(rf_correct_top3) / len(rf_correct_top3)

    results['rf_champion'] = {
        'room_accuracy': -1,
        'artwork_top1_accuracy': rf_acc_top1 * 100,
        'artwork_top3_accuracy': rf_acc_top3 * 100
    }

    y_pred_lr = lr_champion_pipe.predict(X_test_anomalous)
    lr_acc_top1 = accuracy_score(y_test_artwork_encoded, y_pred_lr)

    # Calcola Top-3 per LR
    lr_probs = lr_champion_pipe.predict_proba(X_test_anomalous)
    lr_top3 = lr_probs.argsort(axis=1)[:, -3:]
    lr_correct_top3 = [y_test_artwork_encoded[i] in lr_top3[i] for i in range(len(y_test_artwork_encoded))]
    lr_acc_top3 = sum(lr_correct_top3) / len(lr_correct_top3)

    results['lr_champion'] = {
        'room_accuracy': -1,
        'artwork_top1_accuracy': lr_acc_top1 * 100,
        'artwork_top3_accuracy': lr_acc_top3 * 100
    }

    return results

def plot_accuracy_vs_anomalies(results_data, output_dir='results'):
    
    # Crea la directory di output se non esiste
    os.makedirs(os.path.join(output_dir), exist_ok=True)

    # Prepara i dati per il plotting
    anomalies = sorted(results_data.keys())

    # Crea una figura con 3 subplot impilati verticalmente
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    plt.rcParams.update({
        'font.size': 14,
        'axes.titlesize': 16,
        'axes.labelsize': 15,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'legend.fontsize': 14,
    })

    metrics = ['artwork_top1_accuracy', 'artwork_top3_accuracy']
    titles = ['Artwork Top-1 Accuracy', 'Artwork Top-3 Accuracy']

    colors = {
        'proximity': 'darkblue',
        'knn_k1': 'crimson',
        'knn_k57': 'forestgreen',
        'rf_champion': 'darkorange',
        'lr_champion': 'purple'
    }
    markers = {
        'proximity': 'o',
        'knn_k1': 's',
        'knn_k57': '^',
        'rf_champion': 'D',
        'lr_champion': 'P'
    }

    algorithms_to_plot = ['proximity', 'knn_k1', 'knn_k57', 'rf_champion', 'lr_champion']

    # Liste per memorizzare handle e etichette per la legenda
    legend_handles = []
    legend_labels = []

    for i, (metric, title) in enumerate(zip(metrics, titles)):
        for algorithm in algorithms_to_plot:

            y_values = [results_data[anomaly][algorithm][metric] for anomaly in anomalies]

            if algorithm == 'proximity':
                label = 'Proximity'
            elif algorithm == 'knn_k1':
                label = '1-NN'
            elif algorithm == 'knn_k57':
                label = '57-NN'
            elif algorithm == 'rf_champion':
                label = 'RF Ottimizzato'
            elif algorithm == 'lr_champion':
                label = 'Reg. Logistica'

            line, = axes[i].plot(anomalies, y_values,
                         marker=markers[algorithm],
                         color=colors[algorithm],
                         linewidth=2.5,
                         markersize=8,
                         alpha=0.9)

            if i == 0:
                legend_handles.append(line)
                legend_labels.append(label)

        # Configura il subplot
        axes[i].set_title(title)
        axes[i].set_ylabel('Accuracy (%)')
        axes[i].grid(True, linestyle='--', alpha=0.5, linewidth=0.8)

        # Imposta i limiti dell'asse y
        y_min = 0 # Partiamo da 0
        axes[i].set_ylim(y_min, 100)

        # Aggiungi uno sfondo chiaro per distinguere l'area del grafico
        axes[i].set_facecolor('#f8f9fa')

        # Imposta l'asse x per mostrare solo valori interi
        axes[i].xaxis.set_major_locator(MaxNLocator(integer=True))

    # Imposta un'etichetta x comune
    axes[1].set_xlabel('Number of Malfunctioning Beacons') # MODIFICA: L'indice è 1 ora

    # Aggiungi una singola legenda in fondo alla figura
    fig.legend(legend_handles, legend_labels,
               loc='lower center',
               bbox_to_anchor=(0.5, -0.05), # Regolato per 5 elementi
               ncol=5, # MODIFICA: 5 colonne
               frameon=True,
               fancybox=True,
               shadow=True)

    # Regola lo spazio tra i subplot e aggiungi spazio in fondo per la legenda
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.25, bottom=0.15) # MODIFICA: Aumentato spazio inferiore

    output_path = "/content/drive/My Drive/Tesi Magistrale/Analisi_Robustezza_Beacon.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')

    print(f"Grafico salvato in: {output_path}")

    plt.close(fig)

def save_results_to_csv(results_data, output_dir='results'):
    # Crea la directory di output se non esiste
    os.makedirs(os.path.join(output_dir), exist_ok=True)

    # Prepara i dati per ogni algoritmo
    # MODIFICA: Aggiunti nuovi algoritmi
    algorithms = ['proximity', 'knn_k1', 'knn_k57', 'rf_champion', 'lr_champion']
    algorithm_names = ['Proximity', '1-NN', '57-NN', 'RF Ottimizzato', 'Reg. Logistica']
    metrics = ['room_accuracy', 'artwork_top1_accuracy', 'artwork_top3_accuracy']
    metric_names = ['Room Accuracy', 'Artwork Top-1 Accuracy', 'Artwork Top-3 Accuracy']

    # Crea un singolo CSV con tutti i dati
    all_data = []

    for anomaly in sorted(results_data.keys()):
        for alg_idx, algorithm in enumerate(algorithms):
            row = {
                'Malfunctioning_Beacons': anomaly,
                'Algorithm': algorithm_names[alg_idx]
            }

            for metric_idx, metric in enumerate(metrics):
                row[metric_names[metric_idx]] = results_data[anomaly][algorithm][metric]

            all_data.append(row)

    # Converti in DataFrame e salva
    all_df = pd.DataFrame(all_data)
    csv_path = "/content/drive/My Drive/Tesi Magistrale/Analisi_Robustezza_Beacon.csv"
    all_df.to_csv(csv_path, index=False, float_format='%.2f')

    print(f"Risultati CSV salvati in: {csv_path}")

    return all_df

def main():
    print("="*80)
    print("Simulazione Anomalie Beacon")
    print("="*80)

    # Percorsi dei dati per Android
    data_dir = "/content/data/fingerprints"
    train_file = os.path.join(data_dir, "android_session_1_1.csv")
    test_file = os.path.join(data_dir, "android_session_2_2.csv")

    print(f"Caricamento dati di training: {train_file}")
    print(f"Caricamento dati di test: {test_file}")

    # Carica i dati
    train_data, test_data = load_data(train_file, test_file)

    # Ottieni le colonne dei beacon
    label_columns = ['room', 'artwork']
    beacon_columns = [col for col in test_data.columns if col not in label_columns]

    print(f"Numero di beacon: {len(beacon_columns)}")

    #Prepara e Addestra i Modelli Campione
    print("\nPreparazione e addestramento dei modelli campione...")

    # Prepara i dati per Scikit-learn
    X_train = train_data[beacon_columns]
    y_train_room = train_data['room']
    y_train_artwork = train_data['artwork']

    X_test = test_data[beacon_columns]
    y_test_room = test_data['room']
    y_test_artwork = test_data['artwork']

    artwork_encoder = LabelEncoder()
    y_train_artwork_encoded = artwork_encoder.fit_transform(y_train_artwork)
    y_test_artwork_encoded = artwork_encoder.transform(y_test_artwork)

    rf_champion_params = {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}
    rf_champion = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1, **rf_champion_params)

    lr_champion_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(multi_class='multinomial',
                                     solver='lbfgs',
                                     random_state=RANDOM_SEED,
                                     n_jobs=-1,
                                     max_iter=1000))
    ])

    # Addestra i modelli UNA SOLA VOLTA
    start_train_time = time.time()
    print("Addestramento Random Forest...")
    rf_champion.fit(X_train, y_train_artwork_encoded)
    print("Addestramento Regressione Logistica...")
    lr_champion_pipe.fit(X_train, y_train_artwork_encoded)
    end_train_time = time.time()
    print(f"Modelli campione addestrati in {end_train_time - start_train_time:.2f} secondi.")

    # Definisci l'intervallo di anomalie da simulare
    max_anomalies = 20
    anomaly_range = range(0, max_anomalies + 1, 2)  # 0, 2, 4, ..., 20

    # Memorizza i risultati per ogni numero di anomalie
    all_results = {}

    for num_anomalies in anomaly_range:
        print(f"\nSimulazione di {num_anomalies} beacon malfunzionanti...")

        # Esegui gli esperimenti con le anomalie simulate
        results = run_experiment_with_anomalies(
            train_data, test_data, num_anomalies, beacon_columns,
            rf_champion, lr_champion_pipe, # Passa i modelli addestrati
            X_train, y_train_artwork_encoded, y_test_artwork_encoded # Passa i dati
        )

        # Memorizza i risultati
        all_results[num_anomalies] = results

        # Stampa i risultati correnti (solo Artwork Top-1 per brevità)
        print(f"  Proximity (Top-1): {results['proximity']['artwork_top1_accuracy']:.2f}%")
        print(f"  1-NN (Top-1): {results['knn_k1']['artwork_top1_accuracy']:.2f}%")
        print(f"  57-NN (Top-1): {results['knn_k57']['artwork_top1_accuracy']:.2f}%")
        print(f"  RF Ottimizzato (Top-1): {results['rf_champion']['artwork_top1_accuracy']:.2f}%")
        print(f"  Reg. Logistica (Top-1): {results['lr_champion']['artwork_top1_accuracy']:.2f}%")

    # Salva i risultati su CSV
    save_results_to_csv(all_results)

    # Traccia i risultati
    plot_accuracy_vs_anomalies(all_results)

    print("\nSimulazione completata!")

if __name__ == "__main__":
    if os.path.exists('/content/additionals'):
        os.chdir('/content/additionals')
        main()
    else:
        script_dir = os.path.dirname(__file__)
        parent_dir = os.path.join(script_dir, '..')
        if os.path.exists(os.path.join(parent_dir, 'train')):
             os.chdir(script_dir)
             main()
        else:
             print("Errore: Impossibile trovare le directory 'train' o 'data'.")